#  Agente CLI com Tool-use
## LAB-001 — Day-1: Fundamentos de LLM + Tool-use

**Objetivo:** Construir um agente de linha de comando que decide quando chamar ferramentas externas (tool-use) vs responder diretamente (pure-prompt).

**Entrega:** Notebook executado + tabela comparativa pure-prompt vs tool-use.


In [6]:
# Instalação de dependências
!pip install openai pydantic rich python-dotenv -q

from openai import OpenAI
from pydantic import BaseModel, Field
from datetime import datetime
import json
import os
from dotenv import load_dotenv

load_dotenv()

# ==========================================
# CONFIGURAÇÃO GROQ (GRÁTIS!)
# ==========================================
GROQ_API_KEY = "CHAVE"

# Inicializa cliente OpenAI apontando para o Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Modelo recomendado (Llama 3.3 70B - excelente para tool-use)
MODEL = "llama-3.3-70b-versatile"

print(f"Cliente Groq inicializado (GRÁTIS)!")
print(f"Modelo: {MODEL}")
print(f"Base URL: https://api.groq.com/openai/v1")

# Teste rápido
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Diga 'Olá, Wilson!' em 1 frase."}],
    temperature=0.7,
)
print(f"\n🧪 Teste: {response.choices[0].message.content}")

✅ Cliente Groq inicializado (GRÁTIS)!
🤖 Modelo: llama-3.3-70b-versatile
🌐 Base URL: https://api.groq.com/openai/v1

🧪 Teste: Olá, Wilson!


In [7]:
def get_weather(city: str) -> dict:
    """Retorna o clima atual de uma cidade."""
    mock_clima = {
        "São Paulo": {"temp": 18, "condicao": "nublado", "umidade": 72},
        "Rio de Janeiro": {"temp": 28, "condicao": "ensolarado", "umidade": 65},
        "Belo Horizonte": {"temp": 22, "condicao": "parcialmente nublado", "umidade": 58},
    }
    cidade_normalizada = city.title()
    dados = mock_clima.get(cidade_normalizada, {
        "temp": 20, "condicao": "indisponível", "umidade": 0
    })
    dados["cidade"] = cidade_normalizada
    return dados


def calculator(expression: str) -> dict:
    """Calcula expressões matemáticas com precisão exata."""
    try:
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return {"erro": "Expressão inválida"}
        resultado = eval(expression)
        return {"expressao": expression, "resultado": resultado}
    except Exception as e:
        return {"erro": str(e)}


def get_current_time(timezone: str = "UTC") -> dict:
    """Retorna a data e hora atuais."""
    agora = datetime.now()
    return {
        "timezone": timezone,
        "horario": agora.strftime("%H:%M:%S"),
        "data": agora.strftime("%d/%m/%Y"),
        "timestamp": agora.isoformat(),
    }


TOOLS_AVAILABLE = {
    "get_weather": get_weather,
    "calculator": calculator,
    "get_current_time": get_current_time,
}

print(f"🔧 Tools registradas: {list(TOOLS_AVAILABLE.keys())}")

🔧 Tools registradas: ['get_weather', 'calculator', 'get_current_time']


In [8]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Consulta o clima atual de uma cidade. Use quando o usuário perguntar sobre temperatura, clima ou previsão.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nome da cidade (ex: São Paulo, Rio de Janeiro)"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calcula expressões matemáticas com precisão exata. Use para qualquer operação aritmética com números maiores que 2 dígitos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expressão matemática válida (ex: 123*456, 9876/4)"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Retorna a data e hora atuais. Use quando o usuário perguntar que horas são ou qual a data de hoje.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "Fuso horário (default: UTC)"
                    }
                },
                "required": []
            }
        }
    }
]

print(f"📋 Schemas definidos: {len(TOOLS_SCHEMA)} tools")

📋 Schemas definidos: 3 tools


In [13]:
SYSTEM_PROMPT = """Você é um assistente CLI útil e direto.
- Se o usuário perguntar sobre algo que requer dado em tempo real
  (clima, horário, cálculo exato), use a tool apropriada.
- Se for uma pergunta conceitual ou de conversa, responda diretamente.
- Seja conciso e objetivo."""


def clean_message_for_groq(message_dict: dict) -> dict:
    """Remove campos não suportados pelo Groq na mensagem."""
    # Campos que o Groq não aceita quando reenviamos
    campos_para_remover = ['annotations', 'refusal', 'audio', 'function_call']

    cleaned = {}
    for key, value in message_dict.items():
        if key not in campos_para_remover:
            cleaned[key] = value

    # Se tem tool_calls, limpa cada uma também
    if 'tool_calls' in cleaned and cleaned['tool_calls']:
        cleaned_tools = []
        for tc in cleaned['tool_calls']:
            cleaned_tc = {
                'id': tc.get('id'),
                'type': tc.get('type', 'function'),
                'function': tc.get('function', {})
            }
            cleaned_tools.append(cleaned_tc)
        cleaned['tool_calls'] = cleaned_tools

    return cleaned


def call_llm_with_tool_use(messages: list) -> dict:
    """Chama o LLM com suporte a tool-use."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS_SCHEMA,
        tool_choice="auto",
        temperature=0.7,
    )

    choice = response.choices[0]
    message = choice.message

    if message.tool_calls:
        print(f"🔧 Tool call: {message.tool_calls[0].function.name}")

        # CORREÇÃO: Limpa a mensagem antes de adicionar ao histórico
        message_dict = message.model_dump()
        message_dict_clean = clean_message_for_groq(message_dict)
        messages.append(message_dict_clean)

        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args_raw = tool_call.function.arguments

            # Trata caso onde argumentos são None ou string vazia
            if fn_args_raw and fn_args_raw.strip():
                fn_args = json.loads(fn_args_raw)
                if fn_args is None:
                    fn_args = {}
            else:
                fn_args = {}

            print(f"   📥 Args: {fn_args}")

            # Executa a tool
            fn = TOOLS_AVAILABLE[fn_name]
            resultado = fn(**fn_args)
            print(f"   📤 Result: {resultado}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(resultado, ensure_ascii=False),
            })

        segunda_resposta = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
        )
        return {
            "tipo": "tool_use",
            "resposta": segunda_resposta.choices[0].message.content,
            "tool_usada": fn_name,
        }

    return {
        "tipo": "pure_prompt",
        "resposta": message.content,
        "tool_usada": None,
    }

In [ ]:
from rich.console import Console
from rich.panel import Panel

console = Console()

console.print(Panel.fit(
    "🤖 Agente CLI ativo!\nDigite 'sair' para encerrar.\n"
    "Teste: 'Que horas são?', 'Qual o clima em SP?', 'Quanto é 12345 * 67890?'",
    title="Sessão iniciada",
    style="bold green",
))

while True:
    pergunta = input("\n👤 Você: ").strip()

    if pergunta.lower() in ["sair", "exit", "quit"]:
        console.print("\n[bold red]👋 Até mais![/bold red]")
        break

    if not pergunta:
        continue

    resultado = call_llm_with_tool_use([{"role": "user", "content": pergunta}])

    tipo_emoji = "🔧" if resultado["tipo"] == "tool_use" else "💬"
    tipo_label = "TOOL-USE" if resultado["tipo"] == "tool_use" else "PURE-PROMPT"

    console.print(Panel(
        f"[bold]{resultado['resposta']}[/bold]\n\n"
        f"[dim]Tipo: {tipo_label} {tipo_emoji} | Tool: {resultado['tool_usada'] or 'nenhuma'}[/dim]",
        title="🤖 Assistente",
        style="blue",
    ))

╭───────────────────────────── Sessão iniciada ─────────────────────────────╮
│ 🤖 Agente CLI ativo!                                                      │
│ Digite 'sair' para encerrar.                                              │
│ Teste: 'Que horas são?', 'Qual o clima em SP?', 'Quanto é 12345 * 67890?' │
╰───────────────────────────────────────────────────────────────────────────╯


👤 Você: que horas são?
🔧 Tool call: get_current_time
   📥 Args: {}
   📤 Result: {'timezone': 'UTC', 'horario': '19:30:10', 'data': '02/06/2026', 'timestamp': '2026-06-02T19:30:10.160857'}


╭───────────────────────────────────────────────── 🤖 Assistente ─────────────────────────────────────────────────╮
│ São 19:30:10 no horário de UTC no dia 02/06/2026.                                                               │
│                                                                                                                 │
│ Tipo: TOOL-USE 🔧 | Tool: get_current_time                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯